In [8]:
import matplotlib.pyplot as plt
import numpy as np

from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory

from numpy.typing import NDArray
from typing import Tuple, Optional, List

In [22]:
class MapConstructor:
    def __init__(self, channel_topic: str = "tf"):
        self.outer: Optional[NDArray] = None
        self.inner: List[NDArray] = []

        if channel_topic == "tf":
            self.channel_topic = "/tf"
        elif channel_topic == "odometry":
            self.channel_topic = "/mapping_tool/odometry"
        else:
            raise ValueError("channel_topic must be 'tf' or 'odometry'")

    def extract_point(self, point, strip_z_component: bool) -> NDArray:
        """Transform a given point from a ros2 vector object to a NumPy array and return it."""

        vec = np.zeros(3)
        vec[0] = point.x
        vec[1] = point.y

        if not strip_z_component:
            vec[2] = point.z

        return vec

    def load_ros_bag(self, path: str, measurement_type: str, strip_z_component: bool = True) -> None:
        """Load a ros bag file, extract the data and store it. `measurement_type` is 'inner' or 'outer'.
        'inner' can be multiple measurements and will be appended. 'outer' can only be one and will be set or replaced."""

        with open(path, 'rb') as f:
            reader = make_reader(f, decoder_factories=[DecoderFactory()])

            points = []

            for i, (schema, channel, message, msg) in enumerate(
            reader.iter_decoded_messages(topics=[self.channel_topic])):
                if channel.topic == "/tf":
                    position = msg.transforms[-1].transform.translation
                else:
                    position = msg.pose.pose.position

                points.append(self.extract_point(position, strip_z_component))

        points = np.array(points)

        if measurement_type == "outer":
            self.outer = points
        elif measurement_type == "inner":
            self.inner.append(points)
        else:
            raise ValueError("measurement_type must be 'outer' or 'inner'")

        return None

    def average_points(self, points: NDArray, num_points: int) -> NDArray:
        """Loop over a list of points and average `num_points` of them. Returns a list of averaged points."""

        max_index = int(len(points) / num_points)
        averaged_points = np.zeros((max_index, 3))

        for i in range(max_index):
            average = np.mean(points[i * num_points: (i + 1) * num_points], axis=0)
            averaged_points[i, :] = average

        return averaged_points

    def build_averages(self, num_points: int) -> None:
        # Once for outer
        if self.outer is not None:
            self.outer = self.average_points(self.outer, num_points)

        # Then loop over inner
        for i, points in enumerate(self.inner):
            self.inner[i] = self.average_points(points, num_points)

        return None

    def intentify_corners(self):
        """Loop over a line measurement and identify (actual) corner positions. For now maybe focus on 90 degrees
        but should also work in general for all angles not too close to 180 degrees. Returns a list of points"""
        pass

    def split_points_into_consecutive_segments(self):
        """Extract points that describe lines between identified corner points. Returns a list of point sequences."""
        pass

    def get_track_width(self):
        """Get the width of the track (both directions, i.e. width between inner and outer). Assumes equal track width
        everywhere."""
        pass

    def get_segment_track_separation(self):
        """Get the track separation (middle) line for a track segment without corners."""
        pass

    def get_crossing_bounds(self):
        """Get bounds of the segments that are not part of any track segment, i.e. spaces where there is more than one
        way to keep on driving."""
        pass

    def get_tangents_for_track_segment(self, num_adjacent: int):
        """Get the tangents at each point (estimated from 2n + 1 points, width n (num_adjacent) = 1 as default)."""
        pass

In [24]:
mc = MapConstructor()

mc.load_ros_bag("../data/Maps/outer.mcap", "outer")
mc.load_ros_bag("../data/Maps/inner1.mcap", "inner")
mc.load_ros_bag("../data/Maps/inner2.mcap", "inner")

mc.build_averages(50)